In [ ]:
import sys
from pathlib import Path

_root = Path().resolve()
for cand in (_root, *_root.parents):
    if (cand / 'llm_v2').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break

from llm_v2.config_schema import EmbeddingConfig
from llm_v2.models.embedder import Embedder
from llm_v2.benchmark import (
    evaluate_graph,
    load_clustered_graph,
    print_metrics,
    show_node_alignments,
    show_edge_alignments,
)

## Inputs

In [ ]:
PRED_GRAPH = Path('../llm_v2/output/clustered_graph.json')
GT_GRAPH   = Path('final_bench/graph_clustered.json')

SOURCE_TEXT_PATH = Path('final_bench/formated_fragment2.md')
RESOLVED_TEXT_PATH = Path('../llm_v2/output/coreference_resolved.txt')

EMBED_MODEL = 'all-mpnet-base-v2'
EMBED_DEVICE = 'cpu'

NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K = 10

In [ ]:
pred = load_clustered_graph(PRED_GRAPH)
gt   = load_clustered_graph(GT_GRAPH)


if RESOLVED_TEXT_PATH.exists():
    source_text = RESOLVED_TEXT_PATH.read_text(encoding='utf-8')
    text_provenance = f'resolved ({RESOLVED_TEXT_PATH})'
else:
    source_text = SOURCE_TEXT_PATH.read_text(encoding='utf-8')
    text_provenance = f'source   ({SOURCE_TEXT_PATH})'

print(f'Pred : {len(pred.nodes)} nodes, {len(pred.edges)} edges  ({PRED_GRAPH})')
print(f'GT   : {len(gt.nodes)} nodes, {len(gt.edges)} edges  ({GT_GRAPH})')
print(f'Text : {len(source_text)} chars — {text_provenance}')

In [ ]:
embedder = Embedder(EmbeddingConfig(model_name=EMBED_MODEL, device=EMBED_DEVICE))
print(f'Embedder loaded: {EMBED_MODEL} (dim={embedder.dim})')

## Metrics

In [ ]:
metrics = evaluate_graph(
    pred=pred,
    gt=gt,
    source_text=source_text,
    embedder=embedder,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

print_metrics(metrics)
metrics.summary()

## Alignments

In [ ]:
show_node_alignments(pred, gt, metrics, top_k=TOP_K, direction='pred_to_gt')
print()
show_node_alignments(pred, gt, metrics, top_k=TOP_K, direction='gt_to_pred')

In [ ]:
show_edge_alignments(pred, gt, metrics, top_k=TOP_K, direction='pred_to_gt')
print()
show_edge_alignments(pred, gt, metrics, top_k=TOP_K, direction='gt_to_pred')

## Save

In [ ]:
import json

out_path = Path('benchmark_metrics.json')
out_path.write_text(json.dumps(metrics.summary(), ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {out_path.resolve()}')